# Ngày 2 — Đọc hiểu 9 bảng Olist

**WHY:** Hiểu data trước khi code — tránh schema sai, phải làm lại.

**WHAT:** Đọc từng bảng, trả lời checklist (đáp án gợi ý ở cell Markdown cuối — tự verify lại sau khi chạy).

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)


def data_dir() -> Path:
    """Thư mục data (ổn định dù chạy kernel từ repo hay notebooks/)."""
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        candidate = d / "data"
        if candidate.is_dir() and (candidate / "olist_orders_dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy data-platform/data")


DATA_DIR = data_dir()
print("DATA_DIR =", DATA_DIR)

In [ ]:
orders      = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
items       = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
customers   = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
products    = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers     = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
payments    = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
geo         = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

In [ ]:
def explore(df, name):
    print(f"{'=' * 50}")
    print(f"TABLE: {name}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()}")
    print(f"\nSample:\n{df.head(3)}")


for name, df in [
    ("orders", orders),
    ("items", items),
    ("customers", customers),
    ("products", products),
    ("sellers", sellers),
    ("payments", payments),
    ("reviews", reviews),
    ("geo", geo),
    ("translation", translation),
]:
    explore(df, name)

In [ ]:
# Gợi ý: câu hỏi checklist — chạy để tự kiểm chứng (số có thể khác một chút nếu data đổi)
print("orders rows:", len(orders))
print("\norder_status:\n", orders["order_status"].value_counts())
print("\nNull % cao nhất (orders):\n", (orders.isnull().mean() * 100).sort_values(ascending=False).head(5))
print("\nSample timestamp:", orders["order_purchase_timestamp"].iloc[0])

per_order = items.groupby("order_id").size()
print("\nitems: max rows / order_id:", per_order.max(), "mean:", round(per_order.mean(), 3))
print("payment types:", sorted(payments["payment_type"].unique()))
mp = payments.groupby("order_id").size()
print("orders with >1 payment:", (mp > 1).sum())
print("review_score:", reviews["review_score"].min(), "-", reviews["review_score"].max())
print("null review_comment_message %:", round(reviews["review_comment_message"].isnull().mean() * 100, 2))
z = geo["geolocation_zip_code_prefix"]
print("geo rows:", len(geo), "unique zip prefixes:", z.nunique())

---
## Đáp án checklist (đối chiếu sau khi chạy)

### orders
- **Rows:** ~99.441
- **order_status:** delivered (đa số), shipped, canceled, unavailable, invoiced, processing, created, approved — `value_counts()` để xem số lượng.
- **Null nhiều nhất:** thường là `order_delivered_customer_date`, sau đó `order_delivered_carrier_date`, rồi `order_approved_at`.
- **Timestamp:** trong CSV là **chuỗi** dạng `YYYY-MM-DD HH:MM:SS` (và có cột chỉ có ngày) — nên `pd.to_datetime` khi làm pipeline.

### items
- **1 order_id:** có thể **> 1** dòng (nhiều sản phẩm trong cùng đơn); max ~21, mean ~1.14.
- **Giá tiền:** `price` (giá item), `freight_value` (phí vận chuyển dòng đó).

### payments
- **payment_type:** boleto, credit_card, debit_card, voucher, not_defined.
- **1 order nhiều payment:** có — hàng nghìn đơn có `payment_sequential` > 1.

### reviews
- **review_score:** 1–5.
- **review_comment_message null:** ~59% (đa số review không có text).

### geo
- **Rows:** ~1.000.163 (~1M).
- **Vì sao nhiều:** cùng **zip prefix** có nhiều cặp lat/lng (độ phân giải địa lý chi tiết).
- **Zip duplicate:** đúng — **nhiều dòng / một prefix** là bình thường; không phải bảng 1 dòng/zip.

---
**Bước tiếp:** ERD → `notes/olist-schema.md` · Star schema → `notes/dwh-olist-design.md` · Mục lục → `notes/README.md`.